In [1]:
from langchain.agents import create_agent;
from langchain.tools import tool
from langchain_ollama import ChatOllama
@tool
def get_weather(city:str) -> str:
    """Get weather for a city."""
    return f"The weather in {city} is always breezy."

agent = create_agent(
    model=ChatOllama(model='qwen2.5:3b'),
    tools=[get_weather],
)
for chunk in agent.stream({'messages':{'role':'user', 'content':'What is the weather in pune?'}}, stream_mode='updates',):
    # print(chunk)
    for step, data in chunk.items():
        print(f'step: {step}')
        print(f'content: {data['messages'][-1].content_blocks}')

step: model
content: [{'type': 'tool_call', 'id': '287dd1a4-8c15-42d8-a9f5-2d2e858844c2', 'name': 'get_weather', 'args': {'city': 'Pune'}}]
step: tools
content: [{'type': 'text', 'text': 'The weather in Pune is always breezy.'}]
step: model
content: [{'type': 'text', 'text': 'The weather in Pune is described as breezy. Please note that for more detailed and up-to-date information, I would recommend checking a reliable weather forecasting website or app.'}]


In [2]:
for token,metadata in agent.stream({'messages':{'role':'user', 'content':'What is the weather in Pune?'}}, stream_mode='messages'):
    print(f'source: {metadata['langgraph_node']}')
    print(f'token: {token.content_blocks}\n')
    # print(chunk)

source: model
token: [{'type': 'tool_call_chunk', 'id': 'b6a7becc-37ea-4733-8120-28a66a455fb5', 'name': 'get_weather', 'args': '{"city": "Pune"}'}]

source: model
token: []

source: model
token: []

source: tools
token: [{'type': 'text', 'text': 'The weather in Pune is always breezy.'}]

source: model
token: [{'type': 'text', 'text': 'The'}]

source: model
token: [{'type': 'text', 'text': ' weather'}]

source: model
token: [{'type': 'text', 'text': ' in'}]

source: model
token: [{'type': 'text', 'text': ' Pune'}]

source: model
token: [{'type': 'text', 'text': ' is'}]

source: model
token: [{'type': 'text', 'text': ' always'}]

source: model
token: [{'type': 'text', 'text': ' bree'}]

source: model
token: [{'type': 'text', 'text': 'zy'}]

source: model
token: [{'type': 'text', 'text': '.'}]

source: model
token: [{'type': 'text', 'text': ' Is'}]

source: model
token: [{'type': 'text', 'text': ' there'}]

source: model
token: [{'type': 'text', 'text': ' anything'}]

source: model
token:

In [3]:
from langgraph.config import get_stream_writer
model = ChatOllama(model='qwen2.5:3b')
@tool
def get_population(city:str)->str:
    """Get population data for city."""
    writer = get_stream_writer()
    writer(f'Fetching population data for {city}')
    writer(f'Fetched population data for {city}')
    return f"The population of {city} is 20lakhs"
agent = create_agent(
    model=model,
    tools=[get_population]
)
for chunk in agent.stream({'messages':{'role':'user','content':'what\'s the population of Pune?'}},stream_mode='custom'):
    print(chunk)

Fetching population data for Pune
Fetched population data for Pune


In [4]:
for stream_mode,chunk in agent.stream({'messages':{'role':'user','content':'What\'s the population of pune?'}}, stream_mode=['updates','custom']):
    print(f'mode: {stream_mode}')
    print(f'chunk: {chunk}\n')

mode: updates
chunk: {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-28T17:07:41.2231135Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4066242600, 'load_duration': 190875700, 'prompt_eval_count': 152, 'prompt_eval_duration': 1004012600, 'eval_count': 21, 'eval_duration': 2827051800, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019c0593-2c42-7b63-9f9b-38427f01cb90-0', tool_calls=[{'name': 'get_population', 'args': {'city': 'pune'}, 'id': 'ac502153-a8de-4e5f-9ebe-d7a780471680', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 21, 'total_tokens': 173})]}}

mode: custom
chunk: Fetching population data for pune

mode: custom
chunk: Fetched population data for pune

mode: updates
chunk: {'tools': {'messages': [ToolMessage(content='The population of pune is 20lakhs', name='get_population', id='a7b97615-9cd3-4cc7-a89d-fe5709